## Device & Environment Specifications

| Specification | Details |
|---|---|
| **Platform** | Google Colaboratory |
| **Accelerator** | NVIDIA Tesla T4 |
| **GPU Memory** | 15 GB GDDR6 |
| **GPU Architecture** | Turing (SM 7.5), 2560 CUDA cores |
| **Tensor Cores** | 320 (2nd Gen) |
| **FP32 Peak** | ~8.1 TFLOPS |
| **INT8 Peak** | ~130 TOPS |
| **CPU** | Intel Xeon @ ~2.3 GHz (2 vCPUs) |
| **System RAM** | ~12.7 GB |
| **Disk Space** | ~78 GB |
| **OS** | Ubuntu 22.04 LTS (64-bit) |
| **Python Version** | 3.10.x |
| **CUDA Version** | 12.x |
| **Driver Version** | 525.xx (NVIDIA) |
| **PyTorch** | ≥ 2.0 |
| **Transformers** | ≥ 4.35 |
| **Mixed Precision** | AMP (torch.cuda.amp) |
| **Pipeline** | Pretrained → Fine-tune (cosine LR, grad accum) → FASP (perf-protect + bias-prune) → Benchmark |

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate pynvml scipy openpyxl
!pip install -q "scikit-learn>=1.2,<1.9"


In [ ]:
import os, time, gc, warnings, threading, tempfile, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef,
)
from scipy.stats import pearsonr, spearmanr
from torch.cuda.amp import autocast, GradScaler
import openpyxl

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

if DEVICE == 'cuda':
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        print('GPU       :', pynvml.nvmlDeviceGetName(handle))
        print('VRAM      :', round(pynvml.nvmlDeviceGetMemoryInfo(handle).total / 1e9, 2), 'GB')
        print('CUDA      :', torch.version.cuda)
    except Exception as e:
        print('pynvml init failed:', e)

print('PyTorch   :', torch.__version__)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark        = True


In [ ]:
MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'train', 'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'train', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'train', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'train', 'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'CoLA': ('nyu-mll/glue',      'cola', 'train', 'validation',         'sentence',               'label'),
    'MRPC': ('nyu-mll/glue',      'mrpc', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'STSB': ('nyu-mll/glue',      'stsb', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI': ('nyu-mll/glue',      'wnli', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
}

NUM_LABELS = {
    'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2,
    'RTE': 2, 'CoLA': 2, 'MRPC': 2, 'STSB': 1, 'WNLI': 2,
}

BATCH_SIZE       = 8
FINETUNE_BATCH   = 4
GRAD_ACCUM_STEPS = 4
MAX_SAMPLES      = 300
FINETUNE_SAMPLES = None
MAX_LENGTH       = 128
FINETUNE_EPOCHS  = 3
LR               = 2e-5
WARMUP_RATIO     = 0.1
POLL_INTERVAL_S  = 0.01

SPARSITY        = 0.30
GAMMA           = 0.50
N_CALIB_BATCHES = 8



SCALER = GradScaler(enabled=torch.cuda.is_available())
print(f'Config ready — effective fine-tune batch: {FINETUNE_BATCH * GRAD_ACCUM_STEPS}')
print(f'FASP: sparsity={SPARSITY:.0%}, gamma={GAMMA:.0%}')


In [ ]:
NVML_AVAILABLE = False
try:
    import pynvml as _pynvml
    _pynvml.nvmlInit()
    _nvml_handle = _pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
except Exception:
    pass

class PowerSampler:
    def __init__(self):
        self._samples = []; self._running = False; self._thread = None
    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try: self._samples.append(_pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except: pass
            time.sleep(POLL_INTERVAL_S)
    def start(self):
        self._samples = []; self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True); self._thread.start()
    def stop(self):
        self._running = False; self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0

print('PowerSampler ready.')


In [ ]:
def get_texts(batch, text_col):
    if isinstance(text_col, tuple):
        return list(zip(batch[text_col[0]], batch[text_col[1]]))
    return batch[text_col]

def tokenize(tok, texts, labels=None, is_regression=False):
    if isinstance(texts[0], tuple):
        enc = tok([t[0] for t in texts], [t[1] for t in texts],
                  truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    else:
        enc = tok(list(texts), truncation=True, padding=True,
                  max_length=MAX_LENGTH, return_tensors='pt')
    if labels is not None:
        dtype = torch.float if is_regression else torch.long
        enc['labels'] = torch.tensor(labels, dtype=dtype)
    return enc

def memory_mb(model):
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    os.remove(f.name)
    return round(size_mb, 2)

print('Data utilities ready.')


In [ ]:
def finetune(model, tok, ds_cfg, n_labels, ds_name,
             epochs=None, lr=None, desc='Fine-tuning'):
    path, config, train_split, _, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    _epochs = epochs or FINETUNE_EPOCHS
    _lr     = lr or LR

    ds = (load_dataset(path, config, split=train_split)
          if config else load_dataset(path, split=train_split))
    n  = len(ds) if FINETUNE_SAMPLES is None else min(FINETUNE_SAMPLES, len(ds))
    ds = ds.shuffle(seed=42).select(range(n))

    model.to(DEVICE).train()
    optimizer    = torch.optim.AdamW(model.parameters(), lr=_lr, weight_decay=0.01)
    total_steps  = (n // FINETUNE_BATCH // GRAD_ACCUM_STEPS + 1) * _epochs
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    for epoch in range(_epochs):
        total_loss, steps, accum_loss = 0.0, 0, 0.0
        optimizer.zero_grad()
        for i in range(0, n, FINETUNE_BATCH):
            batch = ds[i:i + FINETUNE_BATCH]
            texts = get_texts(batch, text_col)
            enc   = {k: v.to(DEVICE) for k, v in
                     tokenize(tok, texts, batch[label_col], is_regression).items()}
            with autocast(enabled=torch.cuda.is_available()):
                loss = model(**enc).loss / GRAD_ACCUM_STEPS
            SCALER.scale(loss).backward()
            accum_loss += loss.item(); steps += 1
            if steps % GRAD_ACCUM_STEPS == 0:
                SCALER.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                SCALER.step(optimizer); SCALER.update()
                scheduler.step(); optimizer.zero_grad()
                total_loss += accum_loss; accum_loss = 0.0
        print(f'  {desc} epoch {epoch+1}/{_epochs}  '
              f'loss={total_loss/max(steps//GRAD_ACCUM_STEPS,1):.4f}')
    model.eval()
    return model

print('Fine-tune function ready.')


In [ ]:
def _get_encoder_layers(model):
    if hasattr(model, 'bert'):        return list(model.bert.encoder.layer)
    if hasattr(model, 'distilbert'): return list(model.distilbert.transformer.layer)
    if hasattr(model, 'albert'):
        return [l for g in model.albert.encoder.albert_layer_groups
                  for l in g.albert_layers]
    if hasattr(model, 'mobilebert'): return list(model.mobilebert.encoder.layer)
    return []

def _get_attn(layer):
    if hasattr(layer, 'attention') and hasattr(layer.attention, 'self'):
        return layer.attention.self
    if hasattr(layer, 'attention'):
        return layer.attention
    return None

def _head_dim(attn):
    if hasattr(attn, 'attention_head_size'): return attn.attention_head_size
    return attn.query.weight.shape[0] // attn.num_attention_heads


def _compute_head_scores(model, tok, ds_cfg, ds_name):
    path, config, train_split, _, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    ds = (load_dataset(path, config, split=train_split)
          if config else load_dataset(path, split=train_split))
    ds = ds.shuffle(seed=0).select(range(min(N_CALIB_BATCHES * FINETUNE_BATCH, len(ds))))

    model.to(DEVICE).train()
    model.zero_grad()
    n_batches = max(N_CALIB_BATCHES, 1)

    for i in range(0, len(ds), FINETUNE_BATCH):
        batch = ds[i:i + FINETUNE_BATCH]
        enc   = {k: v.to(DEVICE) for k, v in
                 tokenize(tok, get_texts(batch, text_col),
                          batch[label_col], is_regression).items()}
        with autocast(enabled=torch.cuda.is_available()):
            loss = model(**enc).loss / n_batches
        SCALER.scale(loss).backward()

    _dummy = torch.optim.SGD(model.parameters(), lr=0)
    SCALER.unscale_(_dummy)

    bias_scores, perf_scores = [], []
    for layer in _get_encoder_layers(model):
        attn = _get_attn(layer)
        if attn is None: continue
        hd = _head_dim(attn)
        W_v = attn.value.weight
        for h in range(attn.num_attention_heads):
            s, e = h * hd, (h + 1) * hd
            bias_scores.append(W_v.data[s:e].float().norm().item())
            perf_scores.append(W_v.grad[s:e].float().norm().item()
                               if W_v.grad is not None else 0.0)

    model.zero_grad(); model.eval()
    return bias_scores, perf_scores


def fasp_prune(model, tok, ds_cfg, ds_name,
               sparsity=SPARSITY, gamma=GAMMA):
    bias_s, perf_s = _compute_head_scores(model, tok, ds_cfg, ds_name)
    n = len(bias_s)
    if n == 0:
        print('  Warning: no scoreable heads found.'); return model, []

    n_prot  = max(1, int(round(n * gamma)))
    thresh  = np.partition(np.array(perf_s), -n_prot)[-n_prot]
    protected = {i for i, s in enumerate(perf_s) if s >= thresh}

    cands   = sorted([i for i in range(n) if i not in protected],
                     key=lambda i: bias_s[i], reverse=True)
    pruned  = set(cands[:int(round(n * sparsity))])

    idx = 0
    with torch.no_grad():
        for layer in _get_encoder_layers(model):
            attn = _get_attn(layer)
            if attn is None: continue
            hd = _head_dim(attn)
            for h in range(attn.num_attention_heads):
                if idx in pruned:
                    s, e = h * hd, (h + 1) * hd
                    attn.value.weight.data[s:e] = 0.0
                    if attn.value.bias is not None:
                        attn.value.bias.data[s:e] = 0.0
                idx += 1

    print(f'  FASP: pruned {len(pruned)}/{n} heads  '
          f'[sparsity={sparsity:.0%}, gamma={gamma:.0%}]')
    return model, list(pruned)

print('FASP pruning ready.')


In [ ]:
def benchmark(model, tok, ds_cfg, ds_name):
    path, config, _, eval_split, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    ds = (load_dataset(path, config, split=eval_split)
          if config else load_dataset(path, split=eval_split))
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    model.to(DEVICE).eval()
    preds, labels_list, latencies, energies = [], [], [], []
    t_start = time.perf_counter()

    for i in range(0, len(ds), BATCH_SIZE):
        batch   = ds[i:i + BATCH_SIZE]
        enc     = {k: v.to(DEVICE) for k, v in
                   tokenize(tok, get_texts(batch, text_col)).items()}
        sampler = PowerSampler(); sampler.start()
        t0      = time.perf_counter()
        with torch.no_grad(): out = model(**enc)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        elapsed_ms   = (time.perf_counter() - t0) * 1000
        avg_power_mw = sampler.stop()
        n_items = len(batch[label_col])
        latencies.append(elapsed_ms / n_items)
        energies.append((avg_power_mw * elapsed_ms * 1e-3) / n_items)
        if is_regression: preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:             preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels_list.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    latency    = round(float(np.mean(latencies)), 4)
    throughput = round(len(ds) / total_time, 1)
    energy     = round(float(np.mean(energies)), 4)

    if is_regression:
        return {'Accuracy': None, 'F1': None, 'MCC': None,
                'Precision': None, 'Recall': None,
                'Pearson':  round(pearsonr(preds, labels_list)[0]  * 100, 2),
                'Spearman': round(spearmanr(preds, labels_list)[0] * 100, 2),
                'Latency_ms': latency, 'Throughput_sps': throughput, 'Energy_mJ': energy}

    acc  = round(accuracy_score(labels_list, preds) * 100, 2)
    f1   = round(f1_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    prec = round(precision_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    mcc  = round(matthews_corrcoef(labels_list, preds) * 100, 2)
    return {'Accuracy': acc, 'F1': f1, 'MCC': mcc, 'Precision': prec, 'Recall': rec,
            'Pearson': None, 'Spearman': None,
            'Latency_ms': latency, 'Throughput_sps': throughput, 'Energy_mJ': energy}

print('Benchmark function ready.')


In [ ]:
results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'\n{"="*60}')
        print(f'  {ds_name} | {model_name}')
        print(f'{"="*60}')
        tok = pretrained_model = fp32_model = pruned_model = None
        try:
            n_labels = NUM_LABELS[ds_name]
            tok = AutoTokenizer.from_pretrained(hf_id)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token

            pretrained_model = AutoModelForSequenceClassification.from_pretrained(
                hf_id, num_labels=n_labels, ignore_mismatched_sizes=True)

            # ── [1/4] Fine-tune ─────────────────────────────────────
            print('  [1/4] Fine-tuning...')
            fp32_model = pretrained_model; pretrained_model = None
            fp32_model = finetune(fp32_model, tok, ds_cfg, n_labels, ds_name)

            # ── [2/4] FP32 benchmark (records FP32_F1 + FP32_Precision) ─
            print('  [2/4] FP32 benchmark...')
            fp32_m = benchmark(fp32_model, tok, ds_cfg, ds_name)
            fp32_f1   = fp32_m['F1']
            fp32_prec = fp32_m['Precision']

            # ── [3/4] FASP pruning ───────────────────────────────────
            print(f'  [3/4] FASP pruning '
                  f'(sparsity={SPARSITY:.0%}, gamma={GAMMA:.0%})...')
            pruned_model, pruned_heads = fasp_prune(
                copy.deepcopy(fp32_model), tok, ds_cfg, ds_name)

            # ── [4/4] Pruned benchmark ───────────────────────────────
            print('  [4/4] FASP benchmark...')
            pruned_mem = memory_mb(pruned_model)
            pruned_m   = benchmark(pruned_model, tok, ds_cfg, ds_name)

            results.append({
                'Dataset':          ds_name,
                'Model':            model_name,
                'Memory_MB':        pruned_mem,
                'Latency_ms':       pruned_m['Latency_ms'],
                'Accuracy':         pruned_m['Accuracy'],
                'Sparsity':         int(SPARSITY * 100),
                'Energy_mJ':        pruned_m['Energy_mJ'],
                'Throughput_sps':   pruned_m['Throughput_sps'],
                'F1':               pruned_m['F1'],
                'Precision':        pruned_m['Precision'],
                'Recall':           pruned_m['Recall'],
                'Heads_Pruned':     len(pruned_heads),
                # ── new columns ──
                'FP32_F1':          fp32_f1,
                'FP32_Precision':   fp32_prec,
            })
            print('Done.')

        except Exception as e:
            import traceback
            print(f'  ERROR: {e}')
            traceback.print_exc()
        finally:
            for obj in [pretrained_model, fp32_model, pruned_model, tok]:
                try: del obj
                except: pass
            gc.collect()
            if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
# ── Display results table ─────────────────────────────────────
df = pd.DataFrame(results)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)
pd.set_option('display.float_format', '{:.2f}'.format)

for ds in df['Dataset'].unique():
    print(f'\n{"="*10} {ds} {"="*10}')
    sub = df[df['Dataset'] == ds].set_index(['Model'])
    print(sub.to_string())
print()
